<a href="https://colab.research.google.com/github/sokrypton/7.571/blob/main/L8/hierarchical_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌳 Hierarchical Clustering

Same algorithm, one difference — how do you measure distance between clusters?

| Method | Distance between clusters | Tendency |
|---|---|---|
| **Single Linkage** | Minimum (closest pair) | Chains — elongated clusters |
| **Complete Linkage** | Maximum (farthest pair) | Compact, round clusters |
| **Average Linkage (UPGMA)** | Mean of all pairs | Middle ground |

---

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

---
## 1. Visual Intuition — When Does It Matter?

With well-separated round blobs, all three methods give the same answer.  
The differences show up when cluster shapes are **non-spherical**.

In [ ]:
# Two datasets: easy (blobs) and tricky (moons)
X_blobs, y_blobs = make_blobs(n_samples=150, centers=3, cluster_std=1.0, random_state=42)
X_moons, y_moons = make_moons(n_samples=200, noise=0.05, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='Set1', s=20)
axes[0].set_title('Blobs (easy)', fontweight='bold')
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='Set1', s=20)
axes[1].set_title('Moons (tricky)', fontweight='bold'); axes[1].set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
# Dendrograms on blobs — all three look similar
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, method in zip(axes, ['single', 'complete', 'average']):
    Z = linkage(X_blobs, method=method)
    dendrogram(Z, ax=ax, leaf_rotation=90, leaf_font_size=6, no_labels=True)
    ax.set_title(f'{method.capitalize()} Linkage', fontsize=14, fontweight='bold')
    ax.set_ylabel('Distance')
plt.suptitle('Blobs — All Methods Agree', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Dendrograms on moons — now they differ!
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, method in zip(axes, ['single', 'complete', 'average']):
    Z = linkage(X_moons, method=method)
    dendrogram(Z, ax=ax, leaf_rotation=90, leaf_font_size=6, no_labels=True)
    ax.set_title(f'{method.capitalize()} Linkage', fontsize=14, fontweight='bold')
    ax.set_ylabel('Distance')
plt.suptitle('Moons — Methods Disagree!', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### Cut the dendrogram → get clusters

Unlike K-Means, you don't pick K upfront — you build the full tree, then cut it.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, method in zip(axes, ['single', 'complete', 'average']):
    Z = linkage(X_moons, method=method)
    clusters = fcluster(Z, t=2, criterion='maxclust')
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=clusters, cmap='Set1', s=15)
    ax.set_title(f'{method.capitalize()} (K=2)', fontsize=13, fontweight='bold')
    ax.set_aspect('equal')

plt.suptitle('Cluster Assignments on Moons', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()
print('👆 Single linkage chains along the curve and gets it right!')
print('   Complete and average split based on global distance — wrong for this shape.')

---
## 2. Real Data — Animal Phylogeny

This distance matrix comes from immunological comparison of albumin proteins.  
You already know UPGMA from lecture — now let's see how all three methods handle it.

In [ ]:
dm = np.array(
    [[  0, 32, 48, 51, 50, 48, 98,148],
     [ 32,  0, 26, 34, 29, 33, 84,136],
     [ 48, 26,  0, 42, 44, 44, 92,152],
     [ 51, 34, 42,  0, 44, 38, 86,142],
     [ 50, 29, 44, 44,  0, 24, 89,142],
     [ 48, 33, 44, 38, 24,  0, 90,142],
     [ 98, 84, 92, 86, 89, 90,  0,148],
     [148,136,152,142,142,142,148,  0]], dtype=float)

labels = ['Dog', 'Bear', 'Raccoon', 'Weasel', 'Seal', 'SeaLion', 'Cat', 'Monkey']

# scipy needs condensed form (upper triangle, no diagonal)
dm_condensed = squareform(dm)

plt.figure(figsize=(6, 5))
plt.imshow(dm, cmap='Greys')
plt.xticks(range(8), labels, rotation=45, ha='right')
plt.yticks(range(8), labels)
plt.colorbar(label='Distance')
plt.title('Pairwise Distance Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, method in zip(axes, ['single', 'complete', 'average']):
    Z = linkage(dm_condensed, method=method)
    dendrogram(Z, labels=labels, ax=ax, leaf_rotation=45, leaf_font_size=10)
    ax.set_title(f'{method.capitalize()} Linkage', fontsize=14, fontweight='bold')
    ax.set_ylabel('Distance')

plt.suptitle('Animal Phylogeny — Three Linkage Methods', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print('Average linkage = UPGMA — the method you already know from lecture!')
print('Notice the tree topology is the same here, but branch lengths differ.')

---
## 3. UPGMA from Scratch

The algorithm you already know — implemented step by step.  
The only difference between single/complete/average is the `update_dm` step.

In [ ]:
def hierarchical_clustering(dm, labels, method='average'):
    """
    Agglomerative clustering from scratch.
    method: 'single' (min), 'complete' (max), 'average' (UPGMA)
    Returns a list of merge steps: (label_i, label_j, distance)
    """
    dm = np.array(dm, dtype=float)
    labels = list(labels)
    cluster_sizes = {lab: 1 for lab in labels}
    merge_history = []

    while len(labels) > 1:
        n = len(labels)

        # Find the closest pair (upper triangle only)
        min_dist = np.inf
        mi, mj = 0, 1
        for i in range(n):
            for j in range(i + 1, n):
                if dm[i, j] < min_dist:
                    min_dist = dm[i, j]
                    mi, mj = i, j

        merge_history.append((labels[mi], labels[mj], min_dist))
        new_label = f'({labels[mi]}+{labels[mj]})'

        # Compute distances from new cluster to all others
        new_row = []
        ni = cluster_sizes[labels[mi]]
        nj = cluster_sizes[labels[mj]]
        for k in range(n):
            if k == mi or k == mj:
                continue
            #####################################
            # THIS IS THE ONLY LINE THAT CHANGES
            #####################################
            if method == 'single':
                d = min(dm[mi, k], dm[mj, k])
            elif method == 'complete':
                d = max(dm[mi, k], dm[mj, k])
            else:  # average (UPGMA)
                d = (dm[mi, k] * ni + dm[mj, k] * nj) / (ni + nj)
            new_row.append(d)

        # Remove merged rows/cols, add new one
        keep = [k for k in range(n) if k != mi and k != mj]
        dm = dm[np.ix_(keep, keep)]

        # Add new row and column
        new_row = np.array(new_row)
        dm = np.vstack([dm, new_row[np.newaxis, :]])
        dm = np.hstack([dm, np.append(new_row, 0)[:, np.newaxis]])

        # Update labels and sizes
        cluster_sizes[new_label] = ni + nj
        labels = [labels[k] for k in keep] + [new_label]

    return merge_history


# Run on animal data
for method in ['single', 'complete', 'average']:
    print(f'\n--- {method.upper()} LINKAGE ---')
    history = hierarchical_clustering(dm, labels, method=method)
    for i, (a, b, d) in enumerate(history):
        print(f'  Step {i+1}: merge {a} + {b}  (dist={d:.1f})')

---
## 📝 Key Takeaways

| Concept | Detail |
|---|---|
| **Algorithm** | Find closest pair → merge → update distances → repeat |
| **Single linkage** | min distance — can chain, finds irregular shapes |
| **Complete linkage** | max distance — compact, round clusters |
| **Average (UPGMA)** | mean distance — balanced, used in phylogenetics |
| **vs K-Means** | No need to choose K upfront — cut the dendrogram anywhere |
| **Connection** | scipy `average` linkage = UPGMA from lecture |